# CMIP6 to D3 JSON Pipeline
**Sungod Warriors | DSC 106 Project 3**

Run each cell in order. At the end, download the three JSON files and put them in docs/data/.

In [1]:
!pip install gcsfs xarray zarr cftime --quiet

In [2]:
import gcsfs
import xarray as xr
import numpy as np
import json
import os

fs = gcsfs.GCSFileSystem(token='anon')
print('GCS connected')

GCS connected


In [3]:
paths_to_check = [
    'gs://cmip6/CMIP6/CMIP/MPI-M/MPI-ESM1-2-LR/historical/r1i1p1f1/Amon/tas/gn',
    'gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp126/r1i1p1f1/Amon/tas/gn',
    'gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp245/r1i1p1f1/Amon/tas/gn',
    'gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp585/r1i1p1f1/Amon/tas/gn',
]
for p in paths_to_check:
    try:
        versions = fs.ls(p)
        print('OK  ' + p.split('/')[-4] + ' -> ' + versions[-1])
    except Exception as e:
        print('FAIL ' + p + ': ' + str(e))

OK  r1i1p1f1 -> cmip6/CMIP6/CMIP/MPI-M/MPI-ESM1-2-LR/historical/r1i1p1f1/Amon/tas/gn/v20190710
OK  r1i1p1f1 -> cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp126/r1i1p1f1/Amon/tas/gn/v20190710
OK  r1i1p1f1 -> cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp245/r1i1p1f1/Amon/tas/gn/v20190710
OK  r1i1p1f1 -> cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp585/r1i1p1f1/Amon/tas/gn/v20190710


In [4]:
def open_zarr(path):
    versions = fs.ls(path)
    store = fs.get_mapper(versions[-1])
    return xr.open_zarr(store, consolidated=True)

BASE   = 'gs://cmip6/CMIP6'
MODEL  = 'MPI-M/MPI-ESM1-2-LR'
MEMBER = 'r1i1p1f1'
GRID   = 'gn'

def path_for(experiment, variable):
    group = 'CMIP' if experiment == 'historical' else 'ScenarioMIP'
    return BASE + '/' + group + '/' + MODEL + '/' + experiment + '/' + MEMBER + '/Amon/' + variable + '/' + GRID

print('Functions ready')

Functions ready


In [5]:
print('Loading historical...')
hist_tas = open_zarr(path_for('historical', 'tas'))['tas']
hist_pr  = open_zarr(path_for('historical', 'pr'))['pr']

ssps_data = {}
for ssp in ['ssp126', 'ssp245', 'ssp585']:
    print('Loading ' + ssp + '...')
    ssps_data[ssp] = {
        'tas': open_zarr(path_for(ssp, 'tas'))['tas'],
        'pr':  open_zarr(path_for(ssp, 'pr'))['pr'],
    }
print('Done loading')

Loading historical...


I0505 14:43:12.656279  796593 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0505 14:43:12.666252  796609 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)
I0505 14:43:12.666433  796609 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)


Loading ssp126...
Loading ssp245...
Loading ssp585...
Done loading


In [6]:
baseline_tas = hist_tas.sel(time=slice('1981', '2010')).mean('time').compute()
baseline_pr  = hist_pr.sel(time=slice('1981', '2010')).mean('time').compute()

lon_vals  = baseline_tas.lon.values
LON_0_360 = bool(lon_vals.max() > 180)
print('Lon range: ' + str(round(float(lon_vals.min()), 1)) + ' to ' + str(round(float(lon_vals.max()), 1)) + '  (0-360=' + str(LON_0_360) + ')')

Lon range: 0.0 to 358.1  (0-360=True)


In [7]:
REGIONS = {
    'North America':      {'lat': [25,  70], 'lon': [-140,  -60]},
    'Europe':             {'lat': [35,  70], 'lon': [-10,    40]},
    'South Asia':         {'lat': [5,   35], 'lon': [60,    100]},
    'Sub-Saharan Africa': {'lat': [-35, 15], 'lon': [10,     45]},
    'East Asia':          {'lat': [20,  50], 'lon': [100,   145]},
    'Amazon':             {'lat': [-20,  5], 'lon': [-75,   -45]},
}

def regional_slice(da, bounds):
    lat0, lat1 = sorted(bounds['lat'])
    lon0, lon1 = bounds['lon']
    if LON_0_360:
        lon0 = lon0 % 360
        lon1 = lon1 % 360
    return da.sel(lat=slice(lat0, lat1), lon=slice(min(lon0, lon1), max(lon0, lon1)))

print('Regions defined')

Regions defined


In [9]:
os.makedirs('data', exist_ok=True)
timeseries_out = {}

for ssp in ['ssp126', 'ssp245', 'ssp585']:
    print('Processing ' + ssp + '...')

    h_tas = hist_tas.sortby('time').sel(time=slice('1950', '2014'))
    h_pr  = hist_pr.sortby('time').sel(time=slice('1950', '2014'))
    s_tas = ssps_data[ssp]['tas'].sortby('time').sel(time=slice('2015', '2100'))
    s_pr  = ssps_data[ssp]['pr'].sortby('time').sel(time=slice('2015', '2100'))

    full_tas = xr.concat([h_tas, s_tas], dim='time')
    full_pr  = xr.concat([h_pr,  s_pr],  dim='time')

    ann_tas = (full_tas.groupby('time.year').mean('time') - baseline_tas).compute()
    ann_pr  = ((full_pr.groupby('time.year').mean('time') - baseline_pr) * 86400).compute()
    years   = ann_tas.year.values.tolist()

    timeseries_out[ssp] = {}
    w = np.cos(np.deg2rad(ann_tas.lat))

    timeseries_out[ssp]['global'] = {
        'years': years,
        'tas':   np.round(ann_tas.weighted(w).mean(['lat', 'lon']).values, 3).tolist(),
        'pr':    np.round(ann_pr.weighted(w).mean(['lat', 'lon']).values, 3).tolist(),
    }
    for reg, bounds in REGIONS.items():
        r_tas = regional_slice(ann_tas, bounds)
        r_pr  = regional_slice(ann_pr, bounds)
        rw    = np.cos(np.deg2rad(r_tas.lat))
        timeseries_out[ssp][reg] = {
            'years': years,
            'tas':   np.round(r_tas.weighted(rw).mean(['lat', 'lon']).values, 3).tolist(),
            'pr':    np.round(r_pr.weighted(rw).mean(['lat', 'lon']).values, 3).tolist(),
        }

with open('data/timeseries.json', 'w') as f:
    json.dump(timeseries_out, f, separators=(',', ':'))
print('Written: data/timeseries.json')

Processing ssp126...
Processing ssp245...
Processing ssp585...
Written: data/timeseries.json


In [10]:
gridded_out = {}

for ssp in ['ssp126', 'ssp245', 'ssp585']:
    print('Gridded ' + ssp + '...')

    h_tas = hist_tas.sortby('time').sel(time=slice('1950', '2014'))
    h_pr  = hist_pr.sortby('time').sel(time=slice('1950', '2014'))
    s_tas = ssps_data[ssp]['tas'].sortby('time').sel(time=slice('2015', '2100'))
    s_pr  = ssps_data[ssp]['pr'].sortby('time').sel(time=slice('2015', '2100'))

    full_tas = xr.concat([h_tas, s_tas], dim='time')
    full_pr  = xr.concat([h_pr,  s_pr],  dim='time')

    ann_tas = (full_tas.groupby('time.year').mean('time') - baseline_tas)
    ann_pr  = ((full_pr.groupby('time.year').mean('time') - baseline_pr) * 86400)

    gridded_out[ssp] = {}
    for dec in range(1960, 2101, 10):
        dec_tas = ann_tas.sel(year=slice(dec - 4, dec + 5)).mean('year').compute()
        dec_pr  = ann_pr.sel(year=slice(dec - 4, dec + 5)).mean('year').compute()

        STEP = max(1, len(dec_tas.lat) // 36)
        lats = dec_tas.lat.values[::STEP].tolist()
        lons = dec_tas.lon.values[::STEP].tolist()
        if LON_0_360:
            lons = [(l - 360 if l > 180 else l) for l in lons]

        gridded_out[ssp][str(dec)] = {
            'lats': lats,
            'lons': lons,
            'tas':  np.round(dec_tas.values[::STEP, ::STEP], 2).tolist(),
            'pr':   np.round(dec_pr.values[::STEP, ::STEP], 2).tolist(),
        }
    print('  ' + ssp + ' done')

with open('data/gridded.json', 'w') as f:
    json.dump(gridded_out, f, separators=(',', ':'))
print('Written: data/gridded.json')

Gridded ssp126...
  ssp126 done
Gridded ssp245...
  ssp245 done
Gridded ssp585...
  ssp585 done
Written: data/gridded.json


In [12]:
import os
print(os.path.abspath('data/timeseries.json'))
print(os.path.abspath('data/gridded.json'))
print(os.path.abspath('data/regions.json'))

/Users/wanhansun/Downloads/UCSD/DSC106/project03/data/timeseries.json
/Users/wanhansun/Downloads/UCSD/DSC106/project03/data/gridded.json
/Users/wanhansun/Downloads/UCSD/DSC106/project03/data/regions.json
